In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()

import os
api_key = os.getenv("GROQ_API_KEY")
openai_client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

# model="openai/gpt-oss-20b"
model="meta-llama/llama-4-scout-17b-16e-instruct"
# model="gemini-3.5-flash"

In [2]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
doc_names = ["01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"]

In [6]:
from evaluation_utils import llm_structured

In [7]:
usage_input = 0
for doc in doc_names:
    user_prompt = json.dumps(doc)
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=model
    )
    usage_input += usage.input_tokens

print(usage_input / 3)


106.66666666666667


In [8]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [9]:
ground_truth[1]

{'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
len(chunks)

295

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [13]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [18]:
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

chunk_text_index = build_index(chunks)

In [16]:
from minsearch import VectorSearch

from tqdm.auto import tqdm
import numpy as np
from embedder import Embedder

embed = Embedder()
X = []
batch_size = 50

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [c["content"] for c in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

chunk_index = VectorSearch()
chunk_index.fit(X, chunks)


  0%|          | 0/6 [00:00<?, ?it/s]

In [17]:
q = ground_truth[0]["question"]


In [20]:
chunk_text_index.search(q, num_results=5)[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [24]:
vq = embed.encode(q)
chunk_index.search(vq, num_results=5)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [28]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [48]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = chunk_text_index.search(q["question"], num_results=5)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [36]:
compute_relevance_text(ground_truth[0])

[0, 0, 0, 0, 1]

In [49]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(q["question"], num_results=5)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [50]:
compute_relevance(ground_truth[0], chunk_text_index.search)

[0, 0, 0, 0, 1]

In [51]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [44]:
relevance_total = compute_relevance_total(ground_truth, chunk_text_index.search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [45]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [46]:
hit_rate(relevance_total)

0.7583333333333333

In [52]:
relevance_total = compute_relevance_total([{"filename":q["filename"], "question": embed.encode(q["question"])} for q in ground_truth], chunk_index.search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [53]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [54]:
mrr(relevance_total)

0.5486111111111112

In [55]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [57]:
results = []
for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda q, num_results=5: rrf(
            [chunk_index.search(embed.encode(q), num_results=num_results), chunk_text_index.search(q, num_results=num_results)],
            k
        )
    )
    
    results.append({
        "rrf_k": k,
        "mrr": result["mrr"],
    })


  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [58]:
results

[{'rrf_k': 1, 'mrr': 0.6293981481481483},
 {'rrf_k': 50, 'mrr': 0.6377314814814814},
 {'rrf_k': 100, 'mrr': 0.6377314814814814},
 {'rrf_k': 200, 'mrr': 0.6377314814814814}]